# Kronos GPU Inference Server (Free Colab)

This notebook runs Kronos time-series forecasting on **free Google Colab GPU** and exposes it via a public URL using ngrok.

**Use case:** Jasper Trades backends on 4GB RAM systems can call this API for heavy-duty predictions without using local RAM.

## Setup Instructions

1. Click **Runtime → Change runtime type → GPU**
2. Run all cells
3. Copy the ngrok URL
4. Call `https://YOUR_NGROK_URL/predict` from your Jasper backend

**Session limit:** Colab free tier = 9 hours max per session (reconnect to extend)

In [ ]:
# Step 1: Install dependencies
!pip install torch transformers huggingface_hub safetensors einops flask flask-cors pyngrok nest-asyncio -q

In [ ]:
# Step 2: Download Kronos-mini model
from huggingface_hub import hf_hub_download
import torch
from safetensors.torch import load_file

print("Downloading Kronos-mini model...")
model_path = hf_hub_download(
    repo_id="傲慢的狼队/Kronos",
    filename="kronos-mini.safetensors",
)
print(f"Model downloaded to: {model_path}")

# Load state dict
state_dict = load_file(model_path)
print(f"Loaded {len(state_dict)} tensors")

In [ ]:
# Step 3: Build Kronos-mini model
from torch import nn

class KronosMini(nn.Module):
    """Kronos-mini (4.1M params) for GPU inference"""
    def __init__(self):
        super().__init__()
        self.d_model = 64
        self.nhead = 4
        self.num_layers = 2
        self.dim_feedforward = 128
        self.max_len = 512
        
        self.input_proj = nn.Linear(6, self.d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=self.nhead,
            dim_feedforward=self.dim_feedforward,
            dropout=0.1,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers)
        self.output_proj = nn.Linear(self.d_model, 6)
    
    def forward(self, x):
        embedded = self.input_proj(x)
        encoded = self.encoder(embedded)
        return self.output_proj(encoded)

# Initialize model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = KronosMini().to(device)
model.load_state_dict(state_dict)
model.eval()
print(f"Model loaded on {device}")

In [ ]:
# Step 4: Inference function
import numpy as np

def predict(ohlcv_data, forecast_horizon=50):
    """
    Predict future prices from OHLCV data.
    
    Args:
        ohlcv_data: List of [open, high, low, close, volume, amount] arrays
        forecast_horizon: Number of future bars to predict
    
    Returns:
        Dict with predictions
    """
    import torch
    
    if len(ohlcv_data) < 20:
        raise ValueError(f"Need at least 20 bars, got {len(ohlcv_data)}")
    
    # Convert to tensor
    ohlcv_array = torch.tensor(ohlcv_data, dtype=torch.float32)
    mean = ohlcv_array.mean(dim=0, keepdim=True)
    std = ohlcv_array.std(dim=0, keepdim=True) + 1e-8
    normalized = (ohlcv_array - mean) / std
    
    # Add batch dimension
    input_tensor = normalized.unsqueeze(0).to(device)
    
    # Run inference
    with torch.no_grad():
        predictions_normalized = model(input_tensor)
    
    # Denormalize
    predictions = predictions_normalized[0].cpu() * std + mean
    
    # Extract close predictions (index 3)
    close_predictions = predictions[:, 3][:forecast_horizon].tolist()
    
    # Current price and return
    current_price = ohlcv_data[-1][3]
    predicted_return = (close_predictions[-1] - current_price) / current_price if close_predictions else 0
    
    # Confidence intervals (simplified)
    confidence_lower = [p * 0.98 for p in close_predictions]
    confidence_upper = [p * 1.02 for p in close_predictions]
    
    return {
        "predictions": close_predictions,
        "confidence_lower": confidence_lower,
        "confidence_upper": confidence_upper,
        "predicted_return": predicted_return,
        "current_price": current_price,
        "forecast_horizon": forecast_horizon,
        "status": "success",
    }

In [ ]:
# Step 5: Flask API server
from flask import Flask, request, jsonify
from flask_cors import CORS
import nest_asyncio

nest_asyncio.apply()

app = Flask(__name__)
CORS(app)  # Allow all origins

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "healthy",
        "device": device,
        "model": "kronos-mini",
    })

@app.route("/predict", methods=["POST"])
def predict_api():
    try:
        data = request.json
        ohlcv_data = data.get("ohlcv_data")
        forecast_horizon = data.get("forecast_horizon", 50)
        
        if not ohlcv_data:
            return jsonify({"error": "Missing ohlcv_data"}), 400
        
        result = predict(ohlcv_data, forecast_horizon)
        return jsonify(result)
    
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/predict_batch", methods=["POST"])
def predict_batch_api():
    """Batch predict for multiple symbols"""
    try:
        data = request.json
        symbols_data = data.get("symbols_data", {})
        forecast_horizon = data.get("forecast_horizon", 50)
        
        results = {}
        for symbol, ohlcv in symbols_data.items():
            try:
                results[symbol] = predict(ohlcv, forecast_horizon)
            except Exception as e:
                results[symbol] = {"error": str(e)}
        
        return jsonify(results)
    
    except Exception as e:
        return jsonify({"error": str(e)}), 500

print("Flask API ready")

In [ ]:
# Step 6: Setup ngrok tunnel
from pyngrok import ngrok
import os

# Optional: Set your ngrok authtoken for longer tunnels
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

# Create tunnel
public_url = ngrok.connect(5000)
print(f"\n{'='*50}")
print(f"Kronos GPU Inference API is live!")
print(f"Public URL: {public_url}")
print(f"Endpoints:")
print(f"  GET  {public_url}/health")
print(f"  POST {public_url}/predict")
print(f"  POST {public_url}/predict_batch")
print(f"{'='*50}\n")
print("Keep this notebook running to maintain the tunnel.")
print("Session expires after 9 hours (Colab free tier limit).")

In [ ]:
# Step 7: Start Flask server
print("Starting Flask server on port 5000...")
app.run(port=5000, debug=False, use_reloader=False)